In [ ]:
import pandas as pd
from IPython.display import display
from statsmodels.stats.multitest import multipletests

from study.data import cohort, features, observation_batches, prediction_matrix, prediction_metadata
from study.features import build_features
from study.io import WORK, load, reset_directory, write_table
from study.models import predict
from study.observations import prepare_restrictions, restricted_inputs, restriction_settings, subset_inputs
from study.statistics import bootstrap_difference, loss_matrix, mean_draw_losses, prediction_rows

In [ ]:
RULES = {'protected_variables': ['Height', 'MechVent'], 'terminal_outage_hours': 12}
CONDITIONS = ['random_count_0.25', 'recent_0.25', 'spread_0.25', 'terminal_outage', 'gap_random_12']
PAIRS = [
    ('recent_0.25', 'random_count_0.25'),
    ('spread_0.25', 'random_count_0.25'),
    ('terminal_outage', 'gap_random_12')
]

In [ ]:
reset_directory(WORK / 'predictions/timing')

source_fits = load(WORK / 'models/source.pkl')

In [ ]:
# Prepare each raw batch once and share restrictions across tiers
modes = [restriction_settings(condition)['mode'] for condition in CONDITIONS]
for source, target in [
    ('carevue', 'metavision'),
    ('metavision', 'mimiciv'),
    ('metavision', 'eicu')
]:
    metadata = cohort(target).sort_index()
    original = features(target).loc[metadata.index]
    supports = ('shared',) if target == 'eicu' else ('original',) if source == 'carevue' else ('original', 'shared')
    protected = [column for column in original if column.split('_')[0] in RULES['protected_variables']]

    # Prepare each batch for timing restrictions
    for batch, inputs in enumerate(observation_batches(target)):
        inputs = subset_inputs(inputs, metadata.index)
        if inputs['stays'].empty:
            continue
        
        prepared = prepare_restrictions(inputs, modes)

        # Use ten draws for random schedules and one for deterministic schedules
        for condition in CONDITIONS:
            seeds = list(range(10)) if condition.startswith(('random_count_', 'gap_random_')) else [77]

            # Rebuild restricted features and restore protected inputs
            for draw, seed in enumerate(seeds):
                settings = restriction_settings(condition, seed)
                rebuilt = build_features(restricted_inputs(prepared, settings, RULES))
                rebuilt.loc[:, protected] = original.loc[rebuilt.index, protected]
                evaluated = prediction_metadata(metadata.loc[rebuilt.index])

                # Score all tiers using their frozen full source models
                rows = []
                for support in supports:
                    for tier in ('R1', 'R2', 'R3', 'R4'):
                        fitted = source_fits['models'][source, 0, support, tier]
                        risk = predict(fitted['model'], rebuilt)
                        rows.append(
                            prediction_rows(
                                evaluated, risk, domain = target, source = source, support = support,
                                tier = tier, repeat = 0, restriction = condition,
                            ).assign(draw = draw)
                        )

                write_table(
                    WORK / 'predictions/timing' / f'{source}_{target}_{condition}_{draw}_{batch}.parquet',
                    pd.concat(rows, ignore_index = True),
                )

        print(source, target, 'batch', batch + 1, 'complete')

In [ ]:
# Compare measurement timing at matched observation counts across all tiers
rows, interactions = [], []
for source, domain in [
    ('carevue', 'metavision'),
    ('metavision', 'mimiciv'),
    ('metavision', 'eicu')
]:
    supports = ('shared',) if domain == 'eicu' else ('original',) if source == 'carevue' else ('original', 'shared')

    for support in supports:
        metadata = cohort(domain).sort_index()

        # Compare each timing schedule with its matched reference
        for left, right in PAIRS:
            tier_effects = {}

            for tier in ('R1', 'R2', 'R3', 'R4'):
                filters = [
                    ('source', '==', source),
                    ('domain', '==', domain),
                    ('support', '==', support),
                    ('tier', '==', tier)
                ]

                # Exclude reserved eICU calibration patients from evaluation
                if domain == 'eicu':
                    filters += [('partition', '==', 'test')]

                # Pair schedules on the same evaluation patients
                candidate = prediction_matrix(
                    'timing', metadata.index, filters + [('restriction', '==', left)]
                ).dropna(how = 'all')
                ids = candidate.index
                paired_metadata = prediction_metadata(metadata.loc[ids])

                # Average draw losses before calculating paired differences
                reference = prediction_matrix('timing', ids, filters + [('restriction', '==', right)])
                difference = mean_draw_losses(
                    loss_matrix(paired_metadata.in_hospital_mortality, candidate)
                ) - mean_draw_losses(loss_matrix(paired_metadata.in_hospital_mortality, reference))
                tier_effects[tier] = difference

                # Estimate paired effects with patient or hospital resampling
                for equal_hospital in [False, True] if domain == 'eicu' else [False]:
                    result = bootstrap_difference(
                        paired_metadata,
                        difference,
                        2000,
                        77,
                        cluster = domain == 'eicu',
                        equal_hospital = equal_hospital,
                    )

                    rows.append(
                        dict(
                            source = source,
                            domain = domain,
                            support = support,
                            tier = tier,
                            contrast = left + '_minus_' + right,
                            **result,
                        )
                    )

            # Test whether timing affects each representation differently from R2
            for tier in ('R1', 'R3', 'R4'):
                for equal_hospital in [False, True] if domain == 'eicu' else [False]:
                    result = bootstrap_difference(
                        paired_metadata,
                        tier_effects[tier] - tier_effects['R2'],
                        2000,
                        77,
                        cluster = domain == 'eicu',
                        equal_hospital = equal_hospital,
                    )

                    interactions.append(
                        dict(
                            source = source,
                            domain = domain,
                            support = support,
                            tier = tier,
                            contrast = left + '_minus_' + right,
                            **result,
                        )
                    )

results, effects = pd.DataFrame(rows), pd.DataFrame(interactions)

# Apply Holm correction within each comparison family
for frame in (results, effects):
    for _, family in frame.groupby(['source', 'domain', 'support', 'weighting']):
        frame.loc[family.index, 'holm_p'] = multipletests(family.p_value, method = 'holm')[1]

write_table(WORK / 'results/timing.parquet', results)
write_table(WORK / 'results/timing_interactions.parquet', effects)

display(results.round(5))